# **Closure Weather Label Construction**

This notebook builds the modeling dataset for the road closure forecasting project.

Starting from manually labeled closure/reopening timestamps and hourly weather data, this notebook:

- converts closure records into time intervals
- maps those intervals onto hourly weather rows
- handles missing reopening times conservatively
- defines event-level closure starts
- creates forecasting targets such as `will_close_in_6h`, `will_close_in_12h`, and `will_close_in_24h`

The goal is to produce a clean hourly dataset for downstream feature selection and modeling.

## Data Sources

This notebook uses two main inputs:

1. **Manual closure data**  
   A manually curated table of closure timestamps and reopening timestamps collected from historical road closure posts.

2. **Hourly weather data**  
   Historical hourly weather observations for the Donner Summit / I-80 region.

These sources are merged at the hourly level to create labels for supervised learning.

## Load Packages and Raw Data

First, load the libraries, utility functions, and raw input files used throughout the notebook.

In [185]:

import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[1]))
from main import utils
import importlib
importlib.reload(utils)
import numpy as np

closure_filepath = "/Users/hudson/Desktop/road_project/facebook/data/manual_closure_or_reopening_times_finished.xlsx"
closure_df = pd.read_excel(closure_filepath)
weather_filepath = "/Users/hudson/Desktop/road_project/weather/historical_weather_date.csv"
weather_df = pd.read_csv(weather_filepath)
weather_df = weather_df.drop(columns=["Unnamed: 0"], errors="ignore")
weather_df["date"] = pd.to_datetime(weather_df["date"], errors="coerce")
print(weather_df["date"].head())
print(weather_df["date"].dtype)
weather_df["date"] = pd.to_datetime(weather_df["date"], errors="coerce").dt.tz_localize(None)


0   2017-02-21 00:00:00+00:00
1   2017-02-21 01:00:00+00:00
2   2017-02-21 02:00:00+00:00
3   2017-02-21 03:00:00+00:00
4   2017-02-21 04:00:00+00:00
Name: date, dtype: datetime64[us, UTC]
datetime64[us, UTC]


## Standardize Timestamps

The closure and weather data come from different sources, so timestamps need to be parsed into a consistent datetime format before merging.

Because the weather data is hourly, all later labeling logic is aligned to hourly time buckets.

In [186]:
closure_filepath = "/Users/hudson/Desktop/road_project/facebook/data/manual_closure_or_reopening_times_finished.xlsx"
weather_filepath = "/Users/hudson/Desktop/road_project/weather/historical_weather_date.csv"

closure_df = pd.read_excel(closure_filepath)
weather_df = pd.read_csv(weather_filepath)

# weather cleanup
weather_df = weather_df.drop(columns=["Unnamed: 0"], errors="ignore")
weather_df["date"] = pd.to_datetime(weather_df["date"], errors="coerce").dt.tz_localize(None)

# closure cleanup
closure_df["timestamp"] = pd.to_datetime(closure_df["timestamp"], errors="coerce")
closure_df["time_reopened"] = pd.to_datetime(closure_df["time_reopened"], errors="coerce")

print("closure_df shape:", closure_df.shape)
print("weather_df shape:", weather_df.shape)
print("\nclosure columns:", closure_df.columns.tolist())
print("\nweather columns:", weather_df.columns.tolist())

closure_df shape: (617, 22)
weather_df shape: (79176, 23)

closure columns: ['Unnamed: 0.1', 'Unnamed: 0', 'facebookId', 'facebookUrl', 'feedbackId', 'likes', 'postId', 'timestamp', 'user/name', 'user/profilePic', 'user/profileUrl', 'viewsCount', 'time', 'text', 'is_wb', 'is_eb', 'is_closure', 'is_reopening', 'is_placing', 'is_lifting', 'URL', 'time_reopened']

weather columns: ['date', 'temperature_2m', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_high', 'cloud_cover_mid', 'wind_direction_100m', 'wind_direction_10m', 'wind_speed_100m', 'wind_speed_10m', 'rain', 'snowfall', 'snow_depth', 'weather_code', 'pressure_msl', 'surface_pressure', 'precipitation', 'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m', 'is_day', 'snow_depth_water_equivalent', 'sunshine_duration']


In [187]:
print(weather_df["date"].head())
print(weather_df["date"].dtype)

print(closure_df[["timestamp", "time_reopened"]].head())

0   2017-02-21 00:00:00
1   2017-02-21 01:00:00
2   2017-02-21 02:00:00
3   2017-02-21 03:00:00
4   2017-02-21 04:00:00
Name: date, dtype: datetime64[us]
datetime64[us]
            timestamp       time_reopened
0 2026-02-20 07:32:55                 NaT
1 2026-02-20 00:13:08                 NaT
2 2026-02-19 10:27:26 2026-02-20 00:10:00
3 2026-02-19 07:04:09 2026-02-20 00:10:00
4 2026-02-18 16:27:05 2026-02-20 00:10:00


## Build Raw Closure Intervals

Each closure record is converted into a start/end interval.

- If a reopening time exists, the interval runs from closure to reopening.
- If a reopening time is missing, the closure hour is treated as confirmed closed and the following 23 hours are treated as ambiguous.

At this stage, intervals are still close to the raw post-level records.

In [188]:
raw_intervals = utils.build_closure_intervals(
    closure_df,
    closure_col="timestamp",
    reopen_col="time_reopened",
    missing_reopen_hours=24
)

print("raw_intervals:", raw_intervals.shape)
raw_intervals.head()

raw_intervals: (617, 3)


,closure_start,closure_end,has_reopening_time
0,2017-02-21 18:56:34,2017-02-21 21:25:00,True
1,2017-02-21 21:25:26,2017-02-22 21:25:26,False
2,2017-02-21 22:12:23,2017-02-22 22:12:23,False
3,2017-02-21 23:17:22,2017-02-22 23:17:22,False
4,2017-02-22 07:51:27,2017-02-23 07:51:27,False


In [189]:
print(raw_intervals["has_reopening_time"].value_counts(dropna=False))
print(raw_intervals[["closure_start", "closure_end"]].head())

has_reopening_time
False    446
True     171
Name: count, dtype: int64
        closure_start         closure_end
0 2017-02-21 18:56:34 2017-02-21 21:25:00
1 2017-02-21 21:25:26 2017-02-22 21:25:26
2 2017-02-21 22:12:23 2017-02-22 22:12:23
3 2017-02-21 23:17:22 2017-02-22 23:17:22
4 2017-02-22 07:51:27 2017-02-23 07:51:27


## Collapse Raw Records into Event-Level Intervals

Raw closure records can contain multiple posts or updates referring to the same real-world closure episode.

To define event-level starts for forecasting, intervals are consolidated using:
- shared reopening/end-time logic
- overlap / near-adjacency merging

This produces a more realistic set of closure events than treating every raw record as a separate event.

In [190]:
for gap in [3, 6, 12, 24]:
    event_intervals_test = utils.build_event_intervals(
        raw_intervals,
        max_gap_hours=gap
    )
    print(f"gap={gap}h -> merged events: {len(event_intervals_test)}")

gap=3h -> merged events: 212
gap=6h -> merged events: 210
gap=12h -> merged events: 202
gap=24h -> merged events: 193


In [191]:
event_intervals = utils.build_event_intervals(raw_intervals, max_gap_hours=12)

## Label Hourly Weather Rows with Closure Status

The raw closure intervals are mapped onto the hourly weather table.

The resulting `closure` label uses three states:

- `1` = confirmed closure
- `0` = confirmed open
- `NA` = ambiguous period after a closure with missing reopening time

The ambiguous rows are kept for transparency during label construction, then removed before supervised modeling.

In [192]:
weather_closure = utils.apply_closure_to_weather(
    weather_df,
    raw_intervals,
    weather_time_col="date",
    closure_label_col="closure"
)

## Remove Ambiguous Rows for Modeling

Rows with ambiguous closure status are excluded from the supervised modeling dataset.

This conservative choice reduces label noise and avoids treating uncertain post-closure periods as either open or closed.

In [193]:
clean_df = weather_closure.dropna(subset=["closure"]).copy()
clean_df = clean_df.sort_values("date").reset_index(drop=True)

## Add Winter Season Labels

Because closures are seasonal and evaluation should respect time structure, each row is assigned to a winter season (for example, `2021-2022`).

This will later support winter-based splitting and temporal model evaluation.

In [194]:
clean_df["year"] = clean_df["date"].dt.year
clean_df["month"] = clean_df["date"].dt.month
clean_df["winter"] = np.where(
    clean_df["month"] >= 10,
    clean_df["year"].astype(str) + "-" + (clean_df["year"] + 1).astype(str),
    (clean_df["year"] - 1).astype(str) + "-" + clean_df["year"].astype(str)
)

## Mark Event-Level Closure Starts

Forecasting targets should be based on the start of a closure event, not every hour during a closure.

Using the event-level interval table, each hourly row is marked with `closure_start = 1` if it corresponds to the beginning of a merged closure event.

In [195]:
clean_df = utils.add_closure_start_column(
    clean_df,
    event_intervals,
    weather_time_col="date",
    start_col="closure_start",
    output_col="closure_start"
)

print("Clean rows:", len(clean_df))
print("Open rows:", (clean_df["closure"] == 0).sum())
print("Closed rows:", (clean_df["closure"] == 1).sum())
print("Closure starts:", clean_df["closure_start"].sum())

print("\nClosure starts by winter:")
print(clean_df.groupby("winter")["closure_start"].sum().sort_index())

Clean rows: 73032
Open rows: 71747
Closed rows: 1285
Closure starts: 202

Closure starts by winter:
winter
2016-2017    22
2017-2018    25
2018-2019    30
2019-2020    18
2020-2021    17
2021-2022    28
2022-2023    27
2023-2024    13
2024-2025    16
2025-2026     6
Name: closure_start, dtype: int64


## Create Forecasting Targets

To support practical trip-planning questions, forecasting targets are constructed for multiple horizons:

- `will_close_in_6h`
- `will_close_in_12h`
- `will_close_in_24h`

For a given horizon, a row is labeled `1` if a closure event starts within the next `k` hours.
Only open rows are used as positive warning examples.

In [196]:
forecast_df = clean_df.copy()

for h in [6, 12, 24]:
    forecast_df = utils.make_future_closure_target(
        forecast_df,
        time_col="date",
        start_col="closure_start",
        closure_col="closure",
        horizon_hours=h,
        open_only=True
    )

## Final Dataset Summary

This section checks the final class balance and event counts after label construction.

These summaries help verify that:
- the closure labels are reasonable
- closure starts are distributed across winters
- the forecast targets have plausible positive rates

In [197]:

for col in ["will_close_in_6h", "will_close_in_12h", "will_close_in_24h"]:
    print(col)
    print(forecast_df[col].value_counts())
    print("rate:", forecast_df[col].mean())
    print()

will_close_in_6h
will_close_in_6h
0    71820
1     1212
Name: count, dtype: int64
rate: 0.016595465001643116

will_close_in_12h
will_close_in_12h
0    70608
1     2424
Name: count, dtype: int64
rate: 0.03319093000328623

will_close_in_24h
will_close_in_24h
0    68220
1     4812
Name: count, dtype: int64
rate: 0.06588892540256326



## Save Cleaned Outputs

Save the cleaned hourly dataset so that later notebooks can focus on feature engineering, model selection, and evaluation without repeating the label-construction pipeline.

In [198]:
# Save cleaned outputs for later notebooks

output_dir = "/Users/hudson/Desktop/road_project/main/data"

import os
os.makedirs(output_dir, exist_ok=True)

# main cleaned datasets
weather_closure.to_csv(f"{output_dir}/weather_closure_labeled.csv", index=False)
clean_df.to_csv(f"{output_dir}/weather_closure_clean.csv", index=False)
forecast_df.to_csv(f"{output_dir}/weather_closure_forecast_targets.csv", index=False)

# interval tables
raw_intervals.to_csv(f"{output_dir}/raw_closure_intervals.csv", index=False)
event_intervals.to_csv(f"{output_dir}/event_closure_intervals.csv", index=False)

## Key Outputs

This notebook produces five reusable artifacts for downstream modeling:

- raw closure intervals
- event-level closure intervals
- hourly weather labeled with closure status
- a clean confirmed-status dataset with ambiguous rows removed
- forecasting targets for 6h, 12h, and 24h closure risk

These saved outputs are used in the next notebook for feature selection and baseline modeling.